# 03. 적분과 궤적 생성

적분은 변화율을 누적해서 상태를 얻는 과정이다.
로봇에서는 속도를 적분해 위치를 얻고, 가속도를 적분해 속도와 위치를 얻는다.

$$x(t)=x(0)+\int_0^t v(\tau)d\tau$$

수치적으로는 작은 시간 간격 $\Delta t$ 로 쪼개서 누적한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 속도 적분으로 위치 복원

시간에 따라 변하는 속도 $v(t)$ 를 Euler/trapezoid 방식으로 적분한다.

In [ ]:
t = np.linspace(0, 10, 501)
dt = t[1] - t[0]
v = 0.8 + 0.4*np.sin(1.3*t) + 0.15*np.cos(3*t)

x_euler = np.zeros_like(t)
x_trap = np.zeros_like(t)
for k in range(len(t)-1):
    x_euler[k+1] = x_euler[k] + v[k] * dt
    x_trap[k+1] = x_trap[k] + 0.5 * (v[k] + v[k+1]) * dt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t, v, color='#534AB7', lw=2)
axes[0].set_title('측정된 선속도 v(t)')
axes[0].set_xlabel('time (s)'); axes[0].set_ylabel('m/s')
axes[0].grid(alpha=0.25)

axes[1].plot(t, x_euler, color='#E85D24', lw=2, label='Euler')
axes[1].plot(t, x_trap, '--', color='#1D9E75', lw=2, label='Trapezoid')
axes[1].set_title('속도 적분으로 위치 추정')
axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('m')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
plt.savefig('assets/03_velocity_integration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'최종 위치 Euler: {x_euler[-1]:.4f} m')
print(f'최종 위치 Trapezoid: {x_trap[-1]:.4f} m')

## 2. Differential Drive Odometry

바퀴/속도 명령에서 로봇의 pose를 적분한다.

$$\dot{x}=v\cos\theta, \quad \dot{y}=v\sin\theta, \quad \dot{\theta}=\omega$$

In [ ]:
def integrate_unicycle(v_cmd, w_cmd, dt):
    pose = np.array([0.0, 0.0, 0.0])
    poses = [pose.copy()]
    for v, w in zip(v_cmd, w_cmd):
        x, y, th = pose
        pose = pose + np.array([v*np.cos(th), v*np.sin(th), w]) * dt
        pose[2] = np.arctan2(np.sin(pose[2]), np.cos(pose[2]))
        poses.append(pose.copy())
    return np.array(poses)

T = 16.0
dt = 0.04
time = np.arange(0, T, dt)
v_cmd = 0.7 + 0.15*np.sin(0.8*time)
w_cmd = 0.55*np.sin(0.45*time)
poses = integrate_unicycle(v_cmd, w_cmd, dt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(poses[:,0], poses[:,1], color='#534AB7', lw=2.5)
axes[0].scatter(poses[0,0], poses[0,1], color='#1D9E75', s=80, label='start')
axes[0].scatter(poses[-1,0], poses[-1,1], color='#E85D24', s=80, label='end')
for idx in range(0, len(poses), 45):
    x, y, th = poses[idx]
    axes[0].arrow(x, y, 0.15*np.cos(th), 0.15*np.sin(th), head_width=0.05, color='gray', alpha=0.55)
axes[0].axis('equal'); axes[0].grid(alpha=0.25); axes[0].legend()
axes[0].set_title('Odometry로 적분한 2D pose')

axes[1].plot(time, v_cmd, label='v', color='#534AB7')
axes[1].plot(time, w_cmd, label='ω', color='#E85D24')
axes[1].set_title('입력 명령')
axes[1].set_xlabel('time (s)')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
plt.savefig('assets/03_odometry.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 최소 jerk 궤적

로봇 관절을 부드럽게 움직일 때 시작/끝 위치, 속도, 가속도를 0으로 맞춘다.
대표적인 5차 다항식:

$$s(\tau)=10\tau^3-15\tau^4+6\tau^5, \quad \tau=t/T$$

In [ ]:
T = 3.0
t = np.linspace(0, T, 300)
tau = t / T
s = 10*tau**3 - 15*tau**4 + 6*tau**5
sdot = (30*tau**2 - 60*tau**3 + 30*tau**4) / T
sddot = (60*tau - 180*tau**2 + 120*tau**3) / T**2

q0, qf = np.deg2rad(10), np.deg2rad(95)
q = q0 + (qf - q0) * s
qdot = (qf - q0) * sdot
qddot = (qf - q0) * sddot

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axes[0].plot(t, np.rad2deg(q), color='#534AB7', lw=2.5); axes[0].set_ylabel('q (deg)')
axes[1].plot(t, np.rad2deg(qdot), color='#1D9E75', lw=2.5); axes[1].set_ylabel('qdot (deg/s)')
axes[2].plot(t, np.rad2deg(qddot), color='#E85D24', lw=2.5); axes[2].set_ylabel('qddot (deg/s²)')
axes[2].set_xlabel('time (s)')
for ax in axes:
    ax.grid(alpha=0.25)
plt.suptitle('최소 jerk 관절 궤적')
plt.tight_layout()
plt.savefig('assets/03_minimum_jerk.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 의미 | 로보틱스 활용 |
|------|------|---------------|
| 수치 적분 | 변화율을 시간에 따라 누적 | odometry, IMU dead reckoning |
| Euler 적분 | 구현이 단순하지만 오차 누적 | 빠른 시뮬레이션 |
| Trapezoid 적분 | 인접 샘플 평균 사용 | 더 안정적인 누적 |
| 최소 jerk | 부드러운 시작/정지 | 관절 궤적 생성 |